# FORC multipanel

Plot multiple FORC distributions from FORCsensei or FORCtool exports.

## Input

Select a folder containing CSV, XYZ or TXT FORC distributions and click **Load folder**. Files with the same basename are read once, in CSV–XYZ–TXT order. Distinct processing runs remain separate. Rectangular grids are retained; scattered data are interpolated at the selected grid resolution. Matching VARIFORC parameters are read when available.

## Figure

Set the number of columns, intensity normalization, colour scale, field limits and annotations. Equal field scale (1:1) is enabled by default. Field values must use consistent SI (T) or cgs (Oe) units. For SI inputs, axis limits and the profile position can be displayed in T or mT; this does not rescale the FORC density.

Open **Specimen names and order** to rename or reorder panels. Set **Position** to a grid cell, counted from left to right and top to bottom. Unassigned positions remain blank, and the control list follows the panel positions. If a position is occupied, the consecutive panels shift forward to the next empty position. Positions range from 1 to 500. The initial order follows the loaded filenames. For each loaded file, enter an optional **Specimen** name to use in the panel title. Overlay matching uses both this name and the original panel name; multiple matching files require selection. Leave it blank to keep the existing panel name (the filename without `_XYZ` and `_forc`). For example, enter `TZ01` for `TZ01-150mT_300ms_1.7mT_forc_XYZ.csv`.

Optional overlays include a coercivity profile at the selected $B_u$ (0 by default) and a hysteresis inset. Use **Size (%)** to resize the inset; its right and bottom margins stay fixed. Choose the colour and whether to show the inset's zero axes. With **Show inset axes** enabled, **Measured field range** marks the measured field extrema and labels the positive maximum in teslas, excluding the inferred branch and plot margins. For independent hysteresis files, select their field units; Oe is converted to the vacuum-equivalent field in T. Original MicroMag FORC fields are converted to T by the reader. Use an independent two-column field/moment file (dark blue by default) or derive the major-loop branches from original MicroMag FORC measurements (red by default). Independent hysteresis files are matched by the specimen name followed by the configured suffix or by a separator (`-`, `_`, `.`) and measurement details, e.g. `TZ01-500ms_500mT_hys`. Original FORC files use the same specimen name, preferring exact matches before names with measurement details. If matching is ambiguous, select a file for each panel in **Choose matching files**; no inset is shown until a file is selected. Alternatively, use `hysteresis_map.csv` (`sample,file`) for explicit matching. A filename prefix alone does not establish specimen identity; check the selected input files. In the original FORC folder, processed XYZ/CSV/TXT exports and VARIFORC reports are ignored; suffixes `_forc` and `_XYZ` are removed for matching (e.g. `TZ01_forc` with `TZ01_forc_XYZ.xyz`). If processed map names have a batch prefix (for example `Auto_`), enter it in **Ignore panel prefix**; displayed panel names remain unchanged. Any branch completed by inversion symmetry is shown dashed; it is inferred, not measured.

## Export

Save multipanel figures as PDF, PNG or SVG. Page geometry follows A4 width; panel dimensions and font sizes adjust to the column count. Edit the `APPEARANCE` dictionary at the start of the code cell for line styles, annotation placement and the inset's fixed margins.

Required packages: NumPy, pandas, SciPy, Matplotlib and ipywidgets. `ipyfilechooser` is optional; folder paths can also be entered manually.


In [ ]:
# Figure appearance
# Rerun the cell after changing these values. Widths are in points;
# inset positions and sizes are fractions of the FORC panel.
APPEARANCE = {
    'profile_color': 'gray',
    'profile_linewidth': 0.9,
    'profile_alpha': 0.9,
    'hysteresis_color': 'darkblue',
    'forc_hysteresis_color': 'red',
    'hysteresis_linewidth': 0.4,
    'hysteresis_axes_color': 'black',
    'hysteresis_axes_linewidth': 0.45,
    'hysteresis_right_gap': 0.04,
    'hysteresis_bottom_gap': 0.03,
    'hysteresis_field_fontsize': 4.5,
    'annotation_position': (0.96, 0.94),           # right edge, top edge
    'annotation_font_scale': 1.0,                 # multiply automatic font size
    'smoothing_gap': 1.6,                         # gap below rho_max, in font sizes
    'annotation_linespacing': 1.15,
    'annotation_background': 'white',
    'annotation_background_alpha': 0.75,
    'zero_line_color': 'gray',
    'zero_line_width': 0.6,
    'zero_line_alpha': 0.7,
}

"""Multipanel plots of processed FORC distributions."""

# Imports

import os
import csv
from html import escape
import re
import glob
import string

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from matplotlib.colors import LinearSegmentedColormap
from scipy.interpolate import griddata

import ipywidgets as widgets
from IPython.display import display
try:
    from ipyfilechooser import FileChooser
except ImportError:
    FileChooser = None


# Matplotlib settings

# Embed TrueType fonts in vector exports.
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = [
    'Arial',
    'Helvetica',
    'DejaVu Sans'
]

plt.ioff()


# FORCinel colour map

def FORCinel_colormap(Z):
    """
    Generates FORCsensei / FORCinel dynamic adaptive colormap
    using 10 dynamic anchors.
    """

    if len(Z) == 0 or np.all(np.isnan(Z)):
        return plt.get_cmap('viridis'), 0.0, 1.0

    z_min = np.nanmin(Z)
    z_max = np.nanmax(Z)

    vmin = (
        -z_max * 0.19
        if np.abs(z_min) <= z_max * 0.19
        else z_min
    )

    vmax = z_max

    denom = vmax - vmin if (vmax - vmin) != 0 else 1.0

    anchors = np.zeros(10)

    anchors[1] = (-0.025 * vmax - vmin) / denom
    anchors[2] = (-0.005 * vmax - vmin) / denom
    anchors[3] = (0.025 * vmax - vmin) / denom
    anchors[4] = (0.19 * vmax - vmin) / denom
    anchors[5] = (0.48 * vmax - vmin) / denom
    anchors[6] = (0.64 * vmax - vmin) / denom
    anchors[7] = (0.80 * vmax - vmin) / denom
    anchors[8] = (0.97 * vmax - vmin) / denom
    anchors[9] = 1.0

    anchors = np.clip(anchors, 0.0, 1.0)

    cdict_base = {

        'red': [
            (0.0,    127/255, 127/255),
            (0.1387, 255/255, 255/255),
            (0.1597, 255/255, 255/255),
            (0.1807, 255/255, 255/255),
            (0.3193, 102/255, 102/255),
            (0.563,  204/255, 204/255),
            (0.6975, 204/255, 204/255),
            (0.8319, 153/255, 153/255),
            (0.9748, 76/255,  76/255),
            (1.0,    76/255,  76/255)
        ],

        'green': [
            (0.0,    127/255, 127/255),
            (0.1387, 255/255, 255/255),
            (0.1597, 255/255, 255/255),
            (0.1807, 255/255, 255/255),
            (0.3193, 178/255, 178/255),
            (0.563,  204/255, 204/255),
            (0.6975, 76/255,  76/255),
            (0.8319, 102/255, 102/255),
            (0.9748, 25/255,  25/255),
            (1.0,    25/255,  25/255)
        ],

        'blue': [
            (0.0,    255/255, 255/255),
            (0.1387, 255/255, 255/255),
            (0.1597, 255/255, 255/255),
            (0.1807, 255/255, 255/255),
            (0.3193, 102/255, 102/255),
            (0.563,  76/255,  76/255),
            (0.6975, 76/255,  76/255),
            (0.8319, 153/255, 153/255),
            (0.9748, 76/255,  76/255),
            (1.0,    76/255,  76/255)
        ]
    }

    cdict = {
        k: [
            (
                anchors[i],
                cdict_base[k][i][1],
                cdict_base[k][i][2]
            )
            for i in range(10)
        ]
        for k in ['red', 'green', 'blue']
    }

    return (
        LinearSegmentedColormap('forc_cmap', cdict),
        vmin,
        vmax
    )


# Interface

dir_chooser = (FileChooser(
    path=os.getcwd(),
    title='<b>FORC data folder</b>',
    show_only_dirs=True
) if FileChooser is not None else widgets.HTML(value=''))

dirpath_w = widgets.Text(
    placeholder='Folder path',
    description='Folder:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='90%')
)


# Layout

ncols_w = widgets.IntSlider(
    value=3,
    min=1,
    max=5,
    step=1,
    description='Columns:',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='200px')
)

unit_w = widgets.Dropdown(
    options=[
        ('SI (Tesla)', 'SI'),
        ('cgs (Oersted)', 'cgs')
    ],
    value='SI',
    description='Units:',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='170px')
)

axis_unit_w = widgets.Dropdown(options=['T', 'mT'], value='mT',
    description='Axis units:', layout=widgets.Layout(width='170px'))
axis_ticks_w = widgets.IntSlider(value=4, min=2, max=8, step=1,
    description='Tick intervals:', continuous_update=False,
    layout=widgets.Layout(width='250px'))
axis_units_note = widgets.HTML(
    'Field limits and profile position use the selected axis units. '
    'Switching T/mT preserves the physical range.')

def field_display_factor():
    return 1000.0 if unit_w.value == 'SI' and axis_unit_w.value == 'mT' else 1.0

def native_field(widget):
    return widget.value / field_display_factor()


grid_res_w = widgets.IntSlider(
    value=150,
    min=50,
    max=300,
    step=25,
    description='Interp. grid:',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='200px')
)


# Normalization and annotations

norm_mode_w = widgets.Dropdown(
    options=[
        ('Absolute', 'raw'),
        ('Normalized (ρ/ρmax)', 'max_norm')
    ],
    value='max_norm',
    description='Intensity:',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='230px')
)

show_rho_max_w = widgets.Checkbox(
    value=True,
    description='Show ρmax',
    style={'description_width': 'initial'}
)

# VARIFORC
show_variforc_w = widgets.Checkbox(
    value=True,
    description='VARIFORC',
    style={'description_width': 'initial'}
)

show_sub_labels_w = widgets.Checkbox(
    value=True,
    description='Panel labels',
    style={'description_width': 'initial'}
)

show_zero_line_w = widgets.Checkbox(
    value=True,
    description='Bu = 0 line',
    style={'description_width': 'initial'}
)

clean_ticks_w = widgets.Checkbox(
    value=True,
    description='Hide inner ticks',
    style={'description_width': 'initial'}
)


# Colour scales

cbar_style_w = widgets.Dropdown(
    options=[
        ('Shared', 'global'),
        ('Individual', 'indiv'),
        ('None', 'none')
    ],
    value='global',
    description='Colorbar:',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='230px')
)

cmap_w = widgets.Dropdown(
    options=[
        'FORCinel',
        'viridis',
        'plasma',
        'seismic',
        'coolwarm'
    ],
    value='FORCinel',
    description='Colormap:',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='180px')
)

scale_mode_w = widgets.Dropdown(
    options=[
        ('Individual', 'indiv'),
        ('Shared', 'shared')
    ],
    value='indiv',
    description='Colour scale:',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='200px')
)

levels_w = widgets.Dropdown(
    options=[30, 50, 100],
    value=50,
    description='Levels:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='130px')
)

show_contours_w = widgets.Checkbox(
    value=False,
    description='Contours',
    style={'description_width': 'initial'}
)


# Field limits

xlim_min_w = widgets.FloatText(
    value=0.0,
    description='Bc Min:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='140px')
)

xlim_max_w = widgets.FloatText(
    value=1.0,
    description='Bc Max:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='140px')
)

ylim_min_w = widgets.FloatText(
    value=-1.0,
    description='Bu Min:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='140px')
)

ylim_max_w = widgets.FloatText(
    value=1.0,
    description='Bu Max:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='140px')
)


# Controls

equal_aspect_w = widgets.Checkbox(
    value=True,
    description='Equal field scale (1:1)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
axes_tabs = widgets.Tab(children=[
    widgets.VBox([
        widgets.HTML('<b>Global field limits (all panels)</b>'),
        widgets.HBox([xlim_min_w, xlim_max_w, ylim_min_w, ylim_max_w],
                     layout=widgets.Layout(flex_flow='row wrap'))
    ]),
    widgets.VBox([
        equal_aspect_w,
        widgets.HTML(
            'Equal field scale (1:1) uses the same field-unit length on both axes.'
        )
    ])
])
axes_tabs.set_title(0, 'Field limits')
axes_tabs.set_title(1, 'Axis scale')

specimen_controls = widgets.VBox([])
specimen_order_message = widgets.HTML(value='')
MAX_PANEL_POSITION = 500
specimen_panel = widgets.Accordion(children=[specimen_controls], selected_index=None)
specimen_panel.set_title(0, 'Specimen names and order')
_order_update = {'active': False}

# Optional overlays. Hysteresis data are read only when enabled.
show_profile_w = widgets.Checkbox(value=False, description='Coercivity profile', indent=False)
profile_bu_w = widgets.FloatText(value=0.0, description='Profile Bu:',
    tooltip='Profile position in Axis units (mT by default; Oe for cgs)')
profile_height_w = widgets.IntSlider(value=25, min=5, max=50, step=1,
                                    description='Height (%)', continuous_update=False)
show_hysteresis_w = widgets.Checkbox(value=False, description='Hysteresis inset', indent=False)
hysteresis_size_w = widgets.IntSlider(value=22, min=20, max=50, step=1,
                                     description='Size (%)', continuous_update=False,
                                     tooltip='Maximum inset width and height as a fraction of each panel; the inset stays square.')
show_hysteresis_axes_w = widgets.Checkbox(value=False, description='Show inset axes', indent=False)
show_hysteresis_range_w = widgets.Checkbox(value=False, description='Measured field range', indent=False)
hysteresis_field_unit_w = widgets.Dropdown(
    options=[('Select units', ''), ('T', 'T'), ('mT', 'mT'), ('Oe (vacuum equivalent)', 'Oe')],
    value='', description='Field units:',
    tooltip='Units in independent hysteresis files; the FORC reader converts original fields to T.',
    layout=widgets.Layout(width='300px'))
hysteresis_color_options = [('Red', 'red'), ('Blue', 'darkblue'), ('Black', 'black')]
hysteresis_color_w = widgets.Dropdown(options=hysteresis_color_options,
    value=APPEARANCE['hysteresis_color'], description='Colour:')
forc_hysteresis_color_w = widgets.Dropdown(options=hysteresis_color_options,
    value=APPEARANCE['forc_hysteresis_color'], description='Colour:')
hysteresis_source_w = widgets.Dropdown(
    options=[('Independent hysteresis file(s)', 'independent'),
             ('Original FORC file(s)', 'forc')],
    value='independent', description='Source:',
    layout=widgets.Layout(width='300px'))
hysteresis_choices_ui = widgets.VBox([])
hysteresis_choices = {}
hysteresis_match_prefix_w = widgets.Text(
    value='', description='Ignore panel prefix:',
    placeholder='e.g. Auto_',
    tooltip='Remove this exact prefix only when matching overlays; panel labels stay unchanged.',
    layout=widgets.Layout(width='95%'))
hysteresis_dir_w = widgets.Text(value='', description='Hys folder:',
    placeholder='Blank: use FORC folder',
    layout=widgets.Layout(width='95%'))
hysteresis_suffix_w = widgets.Text(value='_hys', description='Hys suffix:',
    placeholder='e.g. _hys, .hys',
    layout=widgets.Layout(width='75%'))
hysteresis_chooser = (FileChooser(
    path=os.getcwd(), title='<b>Select hysteresis folder:</b>',
    show_only_dirs=True) if FileChooser is not None else widgets.HTML(value=''))
forc_hysteresis_dir_w = widgets.Text(value='', description='Raw FORC folder:',
    placeholder='Select the folder containing original FORC files',
    layout=widgets.Layout(width='95%'))
forc_hysteresis_chooser = (FileChooser(
    path=os.getcwd(), title='<b>Select original FORC folder:</b>',
    show_only_dirs=True) if FileChooser is not None else widgets.HTML(value=''))

def sync_hysteresis_choice(chooser):
    chosen = (getattr(chooser, 'selected_path', None)
              or getattr(chooser, 'selected', None))
    if chosen:
        hysteresis_dir_w.value = chosen

if FileChooser is not None:
    hysteresis_chooser.register_callback(sync_hysteresis_choice)

def sync_forc_hysteresis_choice(chooser):
    chosen = (getattr(chooser, 'selected_path', None)
              or getattr(chooser, 'selected', None))
    if chosen:
        forc_hysteresis_dir_w.value = chosen

if FileChooser is not None:
    forc_hysteresis_chooser.register_callback(sync_forc_hysteresis_choice)

independent_hysteresis_controls = widgets.VBox([
    hysteresis_color_w, hysteresis_chooser, hysteresis_dir_w, hysteresis_suffix_w,
    hysteresis_field_unit_w])
forc_hysteresis_controls = widgets.VBox([
    forc_hysteresis_color_w, forc_hysteresis_chooser, forc_hysteresis_dir_w,
    widgets.HTML('Use original MicroMag FORC files, not processed FORC maps.')])

def update_hysteresis_source(change=None):
    independent_hysteresis_controls.layout.display = (
        '' if hysteresis_source_w.value == 'independent' else 'none')
    forc_hysteresis_controls.layout.display = (
        '' if hysteresis_source_w.value == 'forc' else 'none')

hysteresis_source_w.observe(update_hysteresis_source, names='value')
update_hysteresis_source()

insets_ui = widgets.VBox([
    widgets.HTML('<b>Optional insets</b>'),
    widgets.HBox([show_profile_w, profile_bu_w, profile_height_w]),
    widgets.HBox([show_hysteresis_w, hysteresis_size_w, show_hysteresis_axes_w]),
    show_hysteresis_range_w,
    hysteresis_source_w, hysteresis_match_prefix_w,
    independent_hysteresis_controls, forc_hysteresis_controls,
    hysteresis_choices_ui,
    widgets.HTML('For explicit matching, use hysteresis_map.csv (sample,file). '
                 'Reload after changing files.')])

load_btn = widgets.Button(
    description='Load folder',
    button_style='primary',
    layout=widgets.Layout(width='140px')
)

plot_out = widgets.Output()


# Export options

fmt_w = widgets.Dropdown(
    options=['pdf', 'png', 'svg'],
    value='pdf',
    description='Format:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='140px')
)

dpi_w = widgets.IntText(
    value=300,
    description='DPI:',
    style={'description_width': '40px'},
    layout=widgets.Layout(width='120px')
)

# Default output folder: input folder.
save_dir_w = widgets.Text(
    value='',
    placeholder='Blank = loaded FORC folder',
    description='Save folder:',
    style={'description_width': '85px'},
    layout=widgets.Layout(width='75%')
)

# Blank name: use the default.
save_name_w = widgets.Text(
    value='',
    placeholder='Blank = automatic name',
    description='File name:',
    style={'description_width': '85px'},
    layout=widgets.Layout(width='55%')
)

save_btn = widgets.Button(
    description='Save figure',
    button_style='success',
    layout=widgets.Layout(width='160px')
)

status_msg = widgets.HTML(value="")


# DIRECTORY SYNCHRONIZATION

def sync_dir_choice(chooser):
    """Synchronize the selected directory across FileChooser versions."""
    chosen = (getattr(chooser, 'selected_path', None)
              or getattr(chooser, 'default_path', None)
              or getattr(chooser, 'path', None))
    if chosen:
        dirpath_w.value = chosen


if FileChooser is not None:
    dir_chooser.register_callback(sync_dir_choice)


# DATA CACHE

folder_store = {
    'loaded': False,
    'data': [],
    'folder_path': ''
}


# Input files

def resolve_dirpath():
    """Use the manually entered folder first, then the file chooser."""
    p = os.path.abspath(os.path.expanduser(dirpath_w.value.strip())) if dirpath_w.value.strip() else ''
    if p and os.path.isdir(p):
        return p
    c_path = (getattr(dir_chooser, 'selected_path', None)
              or getattr(dir_chooser, 'default_path', None)
              or getattr(dir_chooser, 'path', None))
    if c_path and os.path.isdir(c_path):
        return c_path
    return None


# VARIFORC metadata

def parse_variforc_file(filepath):
    """Read VARIFORC smoothing parameters as formatted strings."""

    params = {}

    key_map = {
        'sc0': 'Sc0',
        'sc1': 'Sc1',
        'su0': 'Su0',
        'su1': 'Su1',
        'lambda': 'lambda'
    }

    pattern = re.compile(
        r'^\s*Optimal\s+'
        r'(Sc0|Sc1|Su0|Su1|lambda)'
        r'\s*=\s*'
        r'(.+?)'
        r'\s*$',
        re.IGNORECASE
    )

    try:

        with open(
            filepath,
            'r',
            encoding='utf-8',
            errors='ignore'
        ) as f:

            for line in f:

                match = pattern.match(line)

                if match:

                    original_key = match.group(1)
                    value = match.group(2).strip()

                    canonical_key = key_map[
                        original_key.lower()
                    ]

                    params[canonical_key] = value

    except Exception as exc:

        print(
            f"Could not read VARIFORC file: "
            f"{os.path.basename(filepath)}"
        )

        print(exc)

    return params


# Associate VARIFORC metadata

def find_variforc_file(data_filepath, variforc_lookup):
    """Locate VARIFORC metadata for the selected FORC export."""

    stem = os.path.splitext(
        os.path.basename(data_filepath)
    )[0]

    # Remove XYZ suffix, but deliberately KEEP "_forc"
    # because VARIFORC filenames may contain it.
    stem_no_xyz = re.sub(
        r'_XYZ$',
        '',
        stem,
        flags=re.IGNORECASE
    )

    candidates = []

    # Candidate 1:
    # Sample_forc_XYZ -> Sample_forc_VARIFORC

    candidates.append(
        f"{stem_no_xyz}_VARIFORC".lower()
    )

    # Candidate 2 / 3:
    # Try alternatives with or without "_forc"

    if stem_no_xyz.lower().endswith('_forc'):

        root_without_forc = re.sub(
            r'_forc$',
            '',
            stem_no_xyz,
            flags=re.IGNORECASE
        )

        candidates.append(
            f"{root_without_forc}_VARIFORC".lower()
        )

    else:

        candidates.append(
            f"{stem_no_xyz}_forc_VARIFORC".lower()
        )

    # Remove possible duplicated candidate names
    candidates = list(dict.fromkeys(candidates))

    # Search exact normalized match
    for candidate in candidates:

        if candidate in variforc_lookup:
            return variforc_lookup[candidate]

    return None


# FORC table reader

def _is_number_token(value):
    """Return True when a text token can be converted to a finite float."""
    try:
        return np.isfinite(float(str(value).strip()))
    except Exception:
        return False


def read_forc_table(filepath):
    """Read Bc, Bu and density from CSV, XYZ or TXT exports; retain NaNs."""
    with open(str(filepath), 'r', encoding='utf-8-sig', errors='replace') as handle:
        lines = [line.split('#', 1)[0].strip() for line in handle]
    lines = [line for line in lines if line]
    if not lines:
        raise ValueError('Empty FORC table or no data rows.')

    def split_fields(line, delimiter):
        if delimiter is None:
            return line.split()
        return next(csv.reader([line], delimiter=delimiter, skipinitialspace=True))

    def numeric_xyz(fields):
        if len(fields) < 3:
            return None
        try:
            # Also accept Fortran-style scientific notation in text exports.
            values = [float(s.strip().replace('D', 'E').replace('d', 'e'))
                      if s.strip() else np.nan for s in fields[:3]]
        except ValueError:
            return None
        if not np.all(np.isfinite(values[:2])):
            return None
        if not np.isfinite(values[2]):
            values[2] = np.nan
        return values

    preferred = ',' if str(filepath).lower().endswith('.csv') else None
    delimiters = list(dict.fromkeys([preferred, ',', None, ';']))
    scores = [sum(numeric_xyz(split_fields(line, delim)) is not None
                  for line in lines[:200]) for delim in delimiters]
    delimiter = delimiters[int(np.argmax(scores))]
    rows = []
    for line in lines:
        values = numeric_xyz(split_fields(line, delimiter))
        if values is not None:
            rows.append(values)
    if not rows or not np.any(np.isfinite(np.asarray(rows)[:, 2])):
        raise ValueError('No finite numerical FORC density values found.')
    return pd.DataFrame(rows, columns=['x', 'y', 'z'])


def build_forc_grid(table, resolution):
    """Keep complete exported grids intact; interpolate scattered inputs only."""
    x, y, z = (np.asarray(table[key], dtype=float) for key in ('x', 'y', 'z'))
    x_axis, y_axis = np.unique(x), np.unique(y)
    if len(x_axis) < 2 or len(y_axis) < 2:
        raise ValueError('A FORC map needs at least two distinct values on each field axis.')
    if (len(x_axis) * len(y_axis) == len(table)
            and not table[['x', 'y']].duplicated().any()):
        Xi, Yi = np.meshgrid(x_axis, y_axis)
        Zi = np.full(Xi.shape, np.nan)
        Zi[np.searchsorted(y_axis, y), np.searchsorted(x_axis, x)] = z
        return Xi, Yi, Zi, 'native grid'
    finite = np.isfinite(z)
    if np.count_nonzero(finite) < 3:
        raise ValueError('Too few finite points to interpolate a FORC map.')
    Xi, Yi = np.meshgrid(np.linspace(x.min(), x.max(), resolution),
                         np.linspace(y.min(), y.max(), resolution))
    Zi = griddata((x[finite], y[finite]), z[finite], (Xi, Yi),
                  method='linear', fill_value=np.nan)
    if not np.any(np.isfinite(Zi)):
        raise ValueError('Interpolation produced no finite FORC values.')
    return Xi, Yi, Zi, 'interpolated'


def find_forc_file_groups(folder):
    """Group by exact basename, preferring CSV, then XYZ, then TXT."""
    priorities = {'.csv': 0, '.xyz': 1, '.txt': 2}
    files = [os.path.join(folder, name) for name in os.listdir(folder)
             if os.path.isfile(os.path.join(folder, name))
             and os.path.splitext(name)[1].lower() in priorities]
    variforc_lookup = {os.path.splitext(os.path.basename(path))[0].lower(): path
                       for path in files if re.search(r'_VARIFORC\.txt$', path, re.I)}
    groups = {}
    for path in files:
        stem, extension = os.path.splitext(os.path.basename(path))
        if re.search(r'(?:^|_)(?:VARIFORC|Summary|Overview_error)$', stem, re.I):
            continue
        groups.setdefault(stem, []).append(path)
    ordered_groups = []
    for stem in sorted(groups, key=lambda name: (name.casefold(), name)):
        ordered_groups.append(sorted(groups[stem], key=lambda path:
            (priorities[os.path.splitext(path)[1].lower()], os.path.basename(path))))
    return ordered_groups, variforc_lookup


# Load folder

def on_specimen_order_change(change):
    if _order_update['active'] or not folder_store['loaded']:
        return
    samples = folder_store['data']
    moving = next(sample for sample in samples if sample['order_widget'] is change['owner'])
    position = change['new']
    occupied = {sample['order_widget'].value: sample for sample in samples if sample is not moving}
    free = position
    while free in occupied and free <= MAX_PANEL_POSITION:
        free += 1
    _order_update['active'] = True
    try:
        if position < 1 or free > MAX_PANEL_POSITION:
            change['owner'].value = change['old']
            specimen_order_message.value = (
                f"<span style='color:red;'>Panel positions must be between 1 and "
                f"{MAX_PANEL_POSITION}, including displaced panels.</span>")
            return
        for slot in range(free - 1, position - 1, -1):
            occupied[slot]['order_widget'].value = slot + 1
        samples.sort(key=lambda sample: sample['order_widget'].value)
        specimen_controls.children = specimen_controls.children[:2] + tuple(
            sample['row_widget'] for sample in samples)
        specimen_order_message.value = ''
    finally:
        _order_update['active'] = False
    update_plots()


def load_folder_data(b=None):
    """Load one usable export per basename and retain VARIFORC metadata."""
    folder = resolve_dirpath()
    if not folder:
        with plot_out:
            plot_out.clear_output()
            print('Please select or paste a valid folder path.')
        return
    dirpath_w.value = folder
    previous_samples = {sample['source_file']: (sample['order_widget'].value, sample['specimen_widget'].value)
                        for sample in folder_store['data']
                        if 'specimen_widget' in sample and folder_store['folder_path'] == folder}
    specimen_controls.children = ()
    specimen_order_message.value = ''
    folder_store.update(folder_path=folder, data=[], loaded=False,
                        duplicates=[], load_errors=[])
    save_dir_w.placeholder = f'Default: {folder}'
    save_name_w.placeholder = f'Default: FORC_Multipanel_{ncols_w.value}cols'
    groups, variforc_lookup = find_forc_file_groups(folder)
    parsed = []
    for candidates in groups:
        for filepath in candidates:
            try:
                table = read_forc_table(filepath)
                Xi, Yi, Zi, grid_kind = build_forc_grid(table, grid_res_w.value)
                raw_name = os.path.splitext(os.path.basename(filepath))[0]
                clean_name = re.sub(r'(_XYZ|_forc)+$', '', raw_name, flags=re.I)
                variforc_path = find_variforc_file(filepath, variforc_lookup)
                params = parse_variforc_file(variforc_path) if variforc_path else {}
                raw_max = float(np.nanmax(Zi))
                parsed.append(dict(name=clean_name, source_file=filepath,
                                   Xi=Xi, Yi=Yi, Zi=Zi, grid_kind=grid_kind,
                                   zmin=float(np.nanmin(Zi)), zmax=raw_max,
                                   raw_max=raw_max, variforc=params,
                                   variforc_file=variforc_path))
                omitted = [path for path in candidates if path != filepath]
                if omitted:
                    folder_store['duplicates'].append(dict(selected=filepath, omitted=omitted))
                break
            except Exception as exc:
                # Try another format for this basename before rejecting a sample.
                folder_store['load_errors'].append((filepath, str(exc)))
    if not parsed:
        status_msg.value = "<span style='color:red;'>No usable FORC maps found.</span>"
        with plot_out:
            plot_out.clear_output()
            print('No usable FORC maps found in: ' + folder)
            for path, error in folder_store['load_errors']:
                print(os.path.basename(path) + ': ' + error)
        return

    if len(parsed) > MAX_PANEL_POSITION:
        status_msg.value = (f"<span style='color:red;'>The folder contains more than "
                            f"{MAX_PANEL_POSITION} FORC maps; panel positions are limited "
                            "to this number.</span>")
        return
    retained = {sample['source_file'] for sample in parsed}
    used_positions = {position for path, (position, _) in previous_samples.items()
                      if path in retained}
    next_position = max(used_positions, default=0) + 1
    for sample in parsed:
        previous = previous_samples.get(sample['source_file'])
        if previous:
            position = previous[0]
        else:
            if next_position > MAX_PANEL_POSITION or next_position in used_positions:
                next_position = next(slot for slot in range(1, MAX_PANEL_POSITION + 1)
                                     if slot not in used_positions)
            position = next_position
            used_positions.add(position)
            next_position += 1
        name_widget = widgets.Text(
            value=previous[1] if previous else '',
            placeholder=sample['name'], description='Specimen:',
            continuous_update=False, layout=widgets.Layout(width='300px'))
        order_widget = widgets.IntText(
            value=position, description='Position:',
            continuous_update=False, layout=widgets.Layout(width='140px'))
        name_widget.observe(update_plots, names='value')
        order_widget.observe(on_specimen_order_change, names='value')
        sample['specimen_widget'] = name_widget
        sample['order_widget'] = order_widget
    parsed.sort(key=lambda sample: sample['order_widget'].value)
    for sample in parsed:
        sample['row_widget'] = widgets.HBox(
            [sample['order_widget'], sample['specimen_widget'],
             widgets.HTML(value=escape(os.path.basename(sample['source_file'])))],
            layout=widgets.Layout(flex_flow='row wrap', align_items='center'))
    specimen_controls.children = (widgets.HTML(
        'Set panel positions; unused positions stay blank. Occupied positions shift forward.'),
        specimen_order_message) + tuple(sample['row_widget'] for sample in parsed)
    folder_store['data'] = parsed
    # Set all four limits while observers are suspended by loaded=False.
    xlim_min_w.value = float(min(np.min(sample['Xi']) for sample in parsed)) * field_display_factor()
    xlim_max_w.value = float(max(np.max(sample['Xi']) for sample in parsed)) * field_display_factor()
    ylim_min_w.value = float(min(np.min(sample['Yi']) for sample in parsed)) * field_display_factor()
    ylim_max_w.value = float(max(np.max(sample['Yi']) for sample in parsed)) * field_display_factor()
    folder_store['loaded'] = True

    n_duplicates = sum(len(item['omitted']) for item in folder_store['duplicates'])
    n_params = sum(bool(sample['variforc']) for sample in parsed)
    missing = [sample['name'] for sample in parsed if not sample['variforc']]
    details = ''.join('<li>' + escape(os.path.basename(sample['source_file']))
                      + ' (' + sample['grid_kind'] + ')</li>' for sample in parsed)
    status_msg.value = (
        f"<span style='color:green;'><b>Loaded {len(parsed)} unique FORC maps.</b></span><br>"
        f'Duplicate exports omitted: {n_duplicates} (CSV &gt; XYZ &gt; TXT; fallback if needed).<br>'
        f'VARIFORC parameters: {n_params}/{len(parsed)} maps.'
        + ('<br>Missing VARIFORC: ' + escape(', '.join(missing)) if missing else '')
        + '<details><summary>Loaded files</summary><ul>' + details + '</ul></details>'
    )
    if folder_store['load_errors']:
        errors = ''.join('<li>' + escape(os.path.basename(path) + ': ' + reason)
                         + '</li>' for path, reason in folder_store['load_errors'])
        status_msg.value += ('<details><summary>Rejected files / fallback details</summary><ul>'
                             + errors + '</ul></details>')
    update_plots()


load_btn.on_click(
    load_folder_data
)


# Figure rendering

def coercivity_profile(Xi, Yi, Zi, bu=0.0):
    """Interpolate a Bu profile without extrapolating missing data."""
    x, z = [], []
    for j in range(Xi.shape[1]):
        ok = np.isfinite(Yi[:, j]) & np.isfinite(Zi[:, j])
        yy, zz = Yi[ok, j], Zi[ok, j]
        order = np.argsort(yy); yy, zz = yy[order], zz[order]
        yy, unique = np.unique(yy, return_index=True); zz = zz[unique]
        x.append(float(Xi[0, j]))
        z.append(float(np.interp(bu, yy, zz)) if len(yy) >= 2 and yy[0] <= bu <= yy[-1] else np.nan)
    return np.asarray(x), np.asarray(z)


def read_hysteresis_curve(path):
    """Two-column field/moment VSM text; preserve acquisition order and branches."""
    rows = []
    with open(path, encoding='utf-8-sig', errors='replace') as handle:
        for line in handle:
            parts = re.split(r'[,;\s]+', line.strip())
            if len(parts) != 2:
                continue
            try:
                row = tuple(float(v.replace('D', 'E')) for v in parts)
            except ValueError:
                continue
            if np.all(np.isfinite(row)):
                rows.append(row)
    a = np.asarray(rows, dtype=float)
    if len(a) < 6 or np.ptp(a[:, 0]) == 0 or np.ptp(a[:, 1]) == 0:
        raise ValueError('Need at least six field/moment pairs with a nonzero range')
    d = np.diff(a[:, 0]); d = d[d != 0]
    if not (np.any(d > 0) and np.any(d < 0)):
        raise ValueError('No field reversal: file is not a hysteresis loop')
    return a


def read_forc_hysteresis_loop(path):
    """Derive major-loop branches directly from a MicroMag FORC data file."""
    number = r'[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?'
    pair_pattern = re.compile(r'^\s*(' + number + r')\s*,\s*(' + number + r')(?:\s*,.*)?$')
    with open(path, 'r', encoding='utf-8-sig', errors='replace') as handle:
        lines = handle.read().splitlines()

    data_start = next((i + 1 for i, line in enumerate(lines)
                       if '(T)' in line or ('Field' in line and 'Moment' in line)), None)
    if data_start is None:
        data_start = next((i for i, line in enumerate(lines)
                           if pair_pattern.match(line.strip())), None)
    if data_start is None:
        raise ValueError('Could not locate numeric data in the original FORC file')

    header = lines[:data_start]
    hcal = None
    for line in header:
        match = re.match(r'^\s*HCal\s*(?:=|\s)\s*(' + number + r')\s*$', line, re.I)
        if match:
            hcal = float(match.group(1))
            break
    if hcal is None:
        raise ValueError('The FORC header does not contain the HCal calibration field')

    units_line = next((line.lower() for line in header if 'units of measure' in line.lower()), '')
    field_scale, moment_scale = ((1e-4, 1e-3) if 'cgs' in units_line else (1.0, 1.0))
    hcal *= field_scale
    rows, blank_run = [], 0
    for line in lines[data_start:]:
        text = line.strip()
        if not text:
            blank_run += 1
            continue
        match = pair_pattern.match(text)
        if match:
            rows.append((float(match.group(1)) * field_scale,
                         float(match.group(2)) * moment_scale, blank_run))
        blank_run = 0
    if not rows:
        raise ValueError('No numeric field/moment data found in the original FORC file')

    blocks, current = [], []
    def close_block():
        if (len(current) >= 2 or
                (len(current) == 1 and abs(current[0][0] - hcal) <= 0.002)):
            blocks.append(current[:])
        current.clear()

    for index, (field, moment, blanks) in enumerate(rows):
        next_field = rows[index + 1][0] if index + 1 < len(rows) else None
        if blanks >= 2 and current:
            close_block()
        if current and current[-1][0] - field > 0.05:
            close_block()
        if (current and next_field is not None
                and abs(field - hcal) <= 0.002 and field - next_field >= 0.02):
            close_block()
        current.append((field, moment))
    close_block()

    segments = []
    for block in blocks:
        fields = np.asarray([row[0] for row in block], dtype=float)
        moments = np.asarray([row[1] for row in block], dtype=float)
        if len(fields) >= 2 and abs(fields[0] - hcal) <= 0.002:
            segments.append({'field': fields[:1], 'moment': moments[:1], 'kind': 'cal'})
            fields, moments = fields[1:], moments[1:]
            segments.append({'field': fields, 'moment': moments, 'kind': 'forc'})
        else:
            kind = 'forc' if len(fields) > 1 else 'cal'
            segments.append({'field': fields, 'moment': moments, 'kind': kind})

    calibration = [(i, segment['moment'][0]) for i, segment in enumerate(segments)
                   if segment['kind'] == 'cal']
    if len(calibration) >= 2:
        cal_index, cal_moment = np.asarray(calibration, dtype=float).T
        drift = np.interp(np.arange(len(segments)), cal_index, cal_moment - cal_moment[0])
        for i, segment in enumerate(segments):
            segment['moment'] = segment['moment'] - drift[i]

    curves = [segment for segment in segments if segment['kind'] == 'forc']
    if not curves:
        raise ValueError('No FORC measurement curves found in the original file')

    fields, moments = [], []
    for segment in curves:
        field = segment['field']
        moment = segment['moment'].copy()
        if (len(field) < 2 or len(field) != len(moment)
                or not np.all(np.isfinite(field)) or not np.all(np.isfinite(moment))
                or np.any(np.diff(field) < 0)):
            raise ValueError('Invalid or non-increasing field data in a FORC curve')
        if len(field) >= 4:
            if field[1] != field[2]:
                moment[0] = moment[1] + (moment[2] - moment[1]) * (field[0] - field[1]) / (field[2] - field[1])
            if field[-2] != field[-3]:
                moment[-1] = moment[-2] + (moment[-2] - moment[-3]) * (field[-1] - field[-2]) / (field[-2] - field[-3])
        segment['moment'] = moment
        fields.append(field)
        moments.append(moment)

    field_grid = np.unique(np.concatenate(fields))
    moment_grid = np.full((len(curves), len(field_grid)), np.nan)
    for index, (field, moment) in enumerate(zip(fields, moments)):
        in_range = (field_grid >= field[0]) & (field_grid <= field[-1])
        moment_grid[index, in_range] = np.interp(field_grid[in_range], field, moment)
    with np.errstate(invalid='ignore'):
        ascending_moment = np.nanmin(moment_grid, axis=0)
    valid = np.isfinite(ascending_moment)
    ascending = np.column_stack((field_grid[valid], ascending_moment[valid]))

    starts = sorted(((float(np.min(segment['field'])), float(segment['moment'][0]))
                     for segment in curves), key=lambda pair: pair[0])
    if len(starts) < 2:
        raise ValueError('Insufficient FORC reversal points to derive a branch')
    descending = np.asarray(starts, dtype=float)

    inferred = None
    above_reversal = ascending[:, 0] > descending[-1, 0]
    supported_by_symmetry = ascending[:, 0] <= -ascending[0, 0]
    inferred_mask = above_reversal & supported_by_symmetry
    if np.any(inferred_mask):
        inferred_field = ascending[inferred_mask, 0]
        inferred_moment = -np.interp(
            -inferred_field, ascending[:, 0], ascending[:, 1])
        inferred = np.column_stack((inferred_field, inferred_moment))

    return {'measured': (ascending, descending), 'inferred': inferred}


def load_hysteresis_overlays(samples):
    """Load independent loops or derive loops from matched original FORC files."""
    if not show_hysteresis_w.value:
        hysteresis_choices_ui.children = ()
        hysteresis_choices.clear()
        return {}, []
    from pathlib import Path
    source = hysteresis_source_w.value
    folder_value = (forc_hysteresis_dir_w.value.strip() if source == 'forc'
                    else hysteresis_dir_w.value.strip() or folder_store['folder_path'])
    if not folder_value:
        hysteresis_choices_ui.children = ()
        hysteresis_choices.clear()
        return {}, ['Select a folder containing the original FORC files.']
    root = Path(os.path.expanduser(folder_value))
    warnings, curves, mapping = [], {}, {}
    manifest = root / 'hysteresis_map.csv'
    if manifest.is_file():
        import csv
        with manifest.open(encoding='utf-8-sig', newline='') as handle:
            for row in csv.DictReader(handle):
                key = row['sample'].strip().casefold()
                mapping.setdefault(key, []).append(root / row['file'].strip())
    if source == 'independent':
        suffix = hysteresis_suffix_w.value.strip().casefold()
        if not suffix:
            hysteresis_choices_ui.children = ()
            hysteresis_choices.clear()
            return {}, ['Enter a hysteresis filename suffix (for example _hys).']
        def file_key(path):
            name = path.name.casefold()
            if name.endswith(suffix):
                return name[:-len(suffix)]
            if path.suffix.lower() in ('.csv', '.txt', '.dat'):
                stem = path.stem.casefold()
                if stem.endswith(suffix):
                    return stem[:-len(suffix)]
            return None
        files = sorted(p for p in root.rglob('*')
                       if p.is_file() and file_key(p) is not None) if root.is_dir() else []
    else:
        def file_key(path):
            stem = path.name.casefold()
            if stem.endswith('.forc'):
                stem = stem[:-len('.forc')]
            stem = re.sub(r'_xyz$', '', stem)
            return re.sub(r'_forc$', '', stem)
        processed_extensions = {'.csv', '.xyz', '.txt', '.dat', '.npz', '.json',
                                '.pdf', '.png', '.jpg', '.svg'}
        files = sorted(p for p in root.rglob('*')
                       if p.is_file() and p.name != 'hysteresis_map.csv'
                       and p.suffix.lower() not in processed_extensions
                       and not re.search(r'_(?:VARIFORC|Summary|Overview)(?:_|$)', p.stem, re.I)
                       and file_key(p)) if root.is_dir() else []
    active_choices, choice_rows = {}, []
    for sample in samples:
        name = sample['name']
        specimen = sample.get('specimen', '').strip()
        match_name = name
        match_prefix = hysteresis_match_prefix_w.value.strip()
        if match_prefix and match_name.casefold().startswith(match_prefix.casefold()):
            match_name = match_name[len(match_prefix):]
        match_name = match_name.lstrip('_-. ')
        keys = list(dict.fromkeys(key for key in (specimen.casefold(), match_name.casefold()) if key))
        panel_key = name.casefold()
        if not keys:
            warnings.append(f'{name}: empty specimen name for hysteresis matching')
            continue
        if panel_key in mapping:
            matches = mapping[panel_key]
        else:
            mapped_key = next((key for key in keys if key in mapping), None)
            if mapped_key is not None:
                matches = mapping[mapped_key]
            else:
                matches = []
                for key in keys:
                    exact_matches = [p for p in files if file_key(p) == key]
                    prefix_matches = [p for p in files if any(
                        file_key(p).startswith(key + sep) for sep in ('_', '-', '.'))]
                    candidates = (exact_matches or prefix_matches) if source == 'forc' else exact_matches + prefix_matches
                    for path in candidates:
                        if path not in matches:
                            matches.append(path)
        if len(matches) > 1:
            choice_key = (source, os.path.abspath(root), name, tuple(keys))
            paths = tuple(str(path) for path in matches)
            previous = hysteresis_choices.get(choice_key)
            if previous is None or previous['paths'] != paths:
                options = [('Select a file', '')] + [
                    (os.path.relpath(path, root), str(path)) for path in matches]
                dropdown = widgets.Dropdown(options=options, value='',
                                            layout=widgets.Layout(width='100%'))
                dropdown.observe(update_plots, names='value')
                previous = {'paths': paths, 'widget': dropdown}
            active_choices[choice_key] = previous
            choice_rows.append(widgets.VBox([
                widgets.HTML(value='<b>' + escape(name) + '</b>'), previous['widget']]))
            selected = previous['widget'].value
            matches = [path for path in matches if str(path) == selected]
            if not matches:
                warnings.append(f'{name}: ambiguous files; select one in Choose matching files')
                continue
        if len(matches) != 1:
            label = 'original FORC file' if source == 'forc' else 'hysteresis file'
            warnings.append(f'{name}: ' + (f'no {label} matching {keys!r} in {root}' if not matches
                else f'ambiguous {label}s; use hysteresis_map.csv'))
            continue
        try:
            curves[name] = (read_forc_hysteresis_loop(matches[0])
                            if source == 'forc' else read_hysteresis_curve(matches[0]))
        except Exception as exc:
            warnings.append(f'{name}: {exc}')
    hysteresis_choices.clear()
    hysteresis_choices.update(active_choices)
    hysteresis_choices_ui.children = ((widgets.HTML('<b>Choose matching files</b>'),)
                                      + tuple(choice_rows) if choice_rows else ())
    return curves, warnings


def draw_optional_overlays(ax, sample, curve):
    if show_profile_w.value:
        xx, zz = coercivity_profile(sample['Xi'], sample['Yi'], sample['Zi_visible'], native_field(profile_bu_w))
        lo, hi = sorted(ax.get_xlim()); valid = (xx >= lo) & (xx <= hi) & np.isfinite(zz)
        peak = np.nanmax(zz[valid]) if np.any(valid) else np.nan
        if not np.any(valid):
            print(f"{sample['name']}: no profile data at Bu={profile_bu_w.value:g} in the visible window.")
        if np.isfinite(peak) and peak > 0:
            # Mixed transform: physical Bc and a fraction of the panel height.
            yy = np.where(valid, np.maximum(zz, 0) / peak * profile_height_w.value / 100, np.nan)
            ax.plot(xx, yy, transform=ax.get_xaxis_transform(), color=APPEARANCE['profile_color'],
                    linewidth=APPEARANCE['profile_linewidth'], alpha=APPEARANCE['profile_alpha'], zorder=8, clip_on=True)
    if curve is not None:
        # mpl_toolkits supports Matplotlib versions without Axes.inset_axes.
        from mpl_toolkits.axes_grid1.inset_locator import inset_axes
        from matplotlib.transforms import Bbox
        inset_size = hysteresis_size_w.value / 100
        right = 1 - APPEARANCE['hysteresis_right_gap']
        iax = inset_axes(ax, width='100%', height='100%',
                         bbox_to_anchor=(right - inset_size,
                                         APPEARANCE['hysteresis_bottom_gap'],
                                         inset_size, inset_size),
                         bbox_transform=ax.transAxes, loc='lower left',
                         borderpad=0)
        original_locator = iax.get_axes_locator()
        def square_inset_locator(inset, renderer):
            bbox = original_locator(inset, renderer)
            transform = inset.figure.transFigure
            pixels = transform.transform_bbox(bbox)
            side = min(pixels.width, pixels.height)
            square = Bbox.from_extents(pixels.x1 - side, pixels.y0,
                                       pixels.x1, pixels.y0 + side)
            return transform.inverted().transform_bbox(square)
        iax.set_axes_locator(square_inset_locator)
        iax.set_zorder(9)
        iax.patch.set_alpha(0)
        for spine in iax.spines.values():
            spine.set_visible(False)
        if isinstance(curve, dict):
            measured = curve['measured']
            inferred = curve['inferred']
            curve_color = forc_hysteresis_color_w.value
            field = np.concatenate([branch[:, 0] for branch in measured])
            moment = np.concatenate([branch[:, 1] for branch in measured])
        else:
            measured = (curve,)
            inferred = None
            curve_color = hysteresis_color_w.value
            field, moment = curve.T
        if show_hysteresis_axes_w.value:
            iax.axhline(0, color=APPEARANCE['hysteresis_axes_color'], linewidth=APPEARANCE['hysteresis_axes_linewidth'])
            iax.axvline(0, color=APPEARANCE['hysteresis_axes_color'], linewidth=APPEARANCE['hysteresis_axes_linewidth'])
        for branch in measured:
            iax.plot(branch[:, 0], branch[:, 1], color=curve_color,
                     linewidth=APPEARANCE['hysteresis_linewidth'])
        if inferred is not None:
            iax.plot(inferred[:, 0], inferred[:, 1], color=curve_color,
                     linewidth=APPEARANCE['hysteresis_linewidth'], linestyle='--')
        iax.set_xticks([]); iax.set_yticks([])
        # Symmetric limits preserve the field and moment origin.
        field_values = np.concatenate([field, inferred[:, 0]]) if inferred is not None else field
        moment_values = np.concatenate([moment, inferred[:, 1]]) if inferred is not None else moment
        iax.set_xlim(-1.05*np.max(np.abs(field_values)), 1.05*np.max(np.abs(field_values)))
        iax.set_ylim(-1.05*np.max(np.abs(moment_values)), 1.05*np.max(np.abs(moment_values)))
        if show_hysteresis_axes_w.value and show_hysteresis_range_w.value:
            factor = (1.0 if isinstance(curve, dict) else
                      {'T': 1.0, 'mT': 1e-3, 'Oe': 1e-4}.get(hysteresis_field_unit_w.value))
            if factor is not None:
                field_min, field_max = np.min(field), np.max(field)
                iax.plot([field_min, field_max], [0, 0], linestyle='none', marker='|',
                         markersize=3, markeredgewidth=APPEARANCE['hysteresis_axes_linewidth'],
                         color=APPEARANCE['hysteresis_axes_color'])
                if field_max > 0:
                    iax.annotate(f'{field_max * factor:.3g} T', (field_max, 0),
                                 xytext=(-1, -2), textcoords='offset points',
                                 ha='right', va='top',
                                 fontsize=APPEARANCE['hysteresis_field_fontsize'])


def generate_multipanel_figure():
    """Render A4-width FORC panels with the selected display settings."""

    data = folder_store['data']

    n_samples = len(data)
    occupied_positions = {sample['order_widget'].value - 1 for sample in data}
    last_position = max(occupied_positions) + 1

    ncols = ncols_w.value

    nrows = int(
        np.ceil(
            last_position / ncols
        )
    )


    # A4 layout

    A4_WIDTH_INCHES = 8.27


    cell_height_map = {

        1: 4.0,
        2: 3.0,
        3: 2.1,
        4: 1.7,
        5: 1.4
    }


    cell_height = cell_height_map.get(
        ncols,
        2.0
    )


    fig_height = (
        nrows * cell_height
        + 0.5
    )


    fig = plt.figure(
        figsize=(
            A4_WIDTH_INCHES,
            fig_height
        )
    )


    # Margins

    if cbar_style_w.value == 'global':

        left_margin = 0.08
        right_margin = 0.88
        wspace = 0.14

    elif cbar_style_w.value == 'indiv':

        left_margin = 0.07
        right_margin = 0.95
        wspace = 0.30

    else:

        left_margin = 0.08
        right_margin = 0.96
        wspace = 0.10


    hspace = 0.20

    if equal_aspect_w.value:
        x_span = abs(native_field(xlim_max_w) - native_field(xlim_min_w))
        y_span = abs(native_field(ylim_max_w) - native_field(ylim_min_w))
        if not np.isfinite(x_span + y_span) or x_span == 0 or y_span == 0:
            plt.close(fig)
            raise ValueError('Enter finite, distinct minimum and maximum field limits.')
        slot_width = (A4_WIDTH_INCHES * (right_margin - left_margin)
                      / (ncols + (ncols - 1) * wspace))
        if cbar_style_w.value == 'indiv':
            slot_width *= 1.0 - 0.046 - 0.04
        fig_height = max(1.5, slot_width * y_span / x_span
                         * (nrows + (nrows - 1) * hspace) / (0.94 - 0.06))
        fig.set_size_inches(A4_WIDTH_INCHES, fig_height)


    gs = gridspec.GridSpec(

        nrows,
        ncols,

        figure=fig,

        hspace=hspace,
        wspace=wspace,

        left=left_margin,
        right=right_margin,

        top=0.94,
        bottom=0.06
    )


    # Font sizes

    font_title = max(
        6.0,
        11.0 - ncols * 0.9
    )

    font_label = max(
        5.0,
        9.5 - ncols * 0.8
    )

    font_tick = max(
        4.5,
        8.0 - ncols * 0.7
    )

    font_annot = max(
        4.0,
        7.5 - ncols * 0.6
    )

    lw_contour = max(
        0.15,
        0.6 - ncols * 0.08
    )


    # UNIT LABELS

    if unit_w.value == 'SI':

        xlabel = r'$B_\mathrm{c}$ [' + axis_unit_w.value + ']'
        ylabel = r'$B_\mathrm{u}$ [' + axis_unit_w.value + ']'

        raw_unit_str = (
            r'A m$^2$ T$^{-2}$'
        )

    else:

        xlabel = r'$H_\mathrm{c}$ [Oe]'
        ylabel = r'$H_\mathrm{u}$ [Oe]'

        raw_unit_str = (
            r'emu Oe$^{-2}$'
        )


    if norm_mode_w.value == 'raw':

        zlabel = (
            f'FORC Distribution '
            f'[{raw_unit_str}]'
        )

    else:

        zlabel = (
            r'Normalized FORC Distribution '
            r'$\rho / \rho_{\mathrm{max}}$'
        )


    # Prepare plotted data
    # Colour scaling, normalization and rho_max are calculated ONLY from the
    # currently visible Bc/Bu window. Hidden extrema outside the selected axis
    # limits therefore do not control the displayed FORC colour scale.

    processed_data = []

    x_plot_min = min(native_field(xlim_min_w), native_field(xlim_max_w))
    x_plot_max = max(native_field(xlim_min_w), native_field(xlim_max_w))
    y_plot_min = min(native_field(ylim_min_w), native_field(ylim_max_w))
    y_plot_max = max(native_field(ylim_min_w), native_field(ylim_max_w))

    for sample in data:

        Xi = sample['Xi']
        Yi = sample['Yi']
        Zi_raw = sample['Zi'].copy()

        visible_mask = (
            (Xi >= x_plot_min)
            & (Xi <= x_plot_max)
            & (Yi >= y_plot_min)
            & (Yi <= y_plot_max)
            & np.isfinite(Zi_raw)
        )

        visible_values_raw = Zi_raw[visible_mask]

        if visible_values_raw.size > 0:
            visible_raw_max = float(np.nanmax(visible_values_raw))
            visible_raw_min = float(np.nanmin(visible_values_raw))
        else:
            visible_raw_max = np.nan
            visible_raw_min = np.nan

        if norm_mode_w.value == 'max_norm':

            # Normalize to the positive maximum actually visible in this FORC.
            if (
                np.isfinite(visible_raw_max)
                and visible_raw_max != 0
            ):
                Zi = Zi_raw / visible_raw_max
            else:
                Zi = Zi_raw.copy()

        else:
            Zi = Zi_raw.copy()

        # Mask data outside the plotted field window before contouring.
        Zi_visible = np.where(visible_mask, Zi, np.nan)
        visible_values = Zi_visible[np.isfinite(Zi_visible)]

        if visible_values.size > 0:
            zmin = float(np.nanmin(visible_values))
            zmax = float(np.nanmax(visible_values))
        else:
            zmin = np.nan
            zmax = np.nan

        processed_data.append({

            'name': sample['name'],
            'specimen': sample['specimen_widget'].value.strip(),
            'position': sample['order_widget'].value - 1,

            'Xi': Xi,
            'Yi': Yi,
            'Zi': Zi,
            'Zi_visible': Zi_visible,

            'zmin': zmin,
            'zmax': zmax,

            # Maximum within the plotted window.
            'raw_max': visible_raw_max,
            'raw_min': visible_raw_min,

            # Retain the full-data maximum.
            'full_raw_max': sample['raw_max'],

            'variforc': sample.get(
                'variforc',
                {}
            )
        })


    # Shared colour limits from visible data

    finite_zmax = [
        d['zmax'] for d in processed_data
        if np.isfinite(d['zmax'])
    ]

    finite_zmin = [
        d['zmin'] for d in processed_data
        if np.isfinite(d['zmin'])
    ]

    global_zmax = max(finite_zmax) if finite_zmax else 1.0
    global_zmin = min(finite_zmin) if finite_zmin else 0.0


    last_mappable = None

    alphabet = list(
        string.ascii_lowercase
    )


    # Sample panels

    hysteresis_curves, overlay_warnings = load_hysteresis_overlays(processed_data)
    if (show_hysteresis_w.value and show_hysteresis_axes_w.value and show_hysteresis_range_w.value
            and hysteresis_source_w.value == 'independent' and not hysteresis_field_unit_w.value):
        overlay_warnings.append('Select independent hysteresis field units to show the measured range in T.')
    if overlay_warnings:
        print('Hysteresis insets: ' + '; '.join(overlay_warnings))

    for idx, sample in enumerate(
        processed_data
    ):

        position = sample['position']
        r, c = divmod(position, ncols)


        ax = fig.add_subplot(
            gs[r, c]
        )


        Xi = sample['Xi']
        Yi = sample['Yi']
        Zi = sample['Zi_visible']

        has_visible_data = np.any(np.isfinite(Zi))


        # Colour map

        if cmap_w.value == 'FORCinel':

            if scale_mode_w.value == 'indiv':

                cmap_data = (
                    Zi[np.isfinite(Zi)]
                    if has_visible_data
                    else np.array([0.0, 1.0])
                )

            else:

                cmap_data = np.array([
                    global_zmin,
                    global_zmax
                ])


            cmap, base_vmin, base_vmax = FORCinel_colormap(
                cmap_data
            )

        else:

            cmap = plt.get_cmap(
                cmap_w.value
            )


            if scale_mode_w.value == 'indiv':

                if has_visible_data:
                    base_vmin = sample['zmin']
                    base_vmax = sample['zmax']
                else:
                    base_vmin = 0.0
                    base_vmax = 1.0

            else:

                base_vmin = global_zmin
                base_vmax = global_zmax

        # Avoid degenerate colour ranges for constant/empty windows.
        if not np.isfinite(base_vmin):
            base_vmin = 0.0
        if not np.isfinite(base_vmax):
            base_vmax = 1.0
        if base_vmax == base_vmin:
            eps = max(abs(base_vmax) * 1e-9, 1e-12)
            base_vmin -= eps
            base_vmax += eps


        # Filled contours

        if has_visible_data:

            # Explicit levels keep the contour colours tied to the visible
            # range instead of allowing hidden values to influence them.
            contour_levels = np.linspace(
                base_vmin,
                base_vmax,
                levels_w.value
            )

            cs = ax.contourf(

                Xi,
                Yi,
                Zi,

                levels=contour_levels,

                cmap=cmap,

                vmin=base_vmin,
                vmax=base_vmax,
                extend='both'
            )

            last_mappable = cs

        else:

            cs = None
            ax.text(
                0.5,
                0.5,
                'No FORC data\nin selected window',
                transform=ax.transAxes,
                ha='center',
                va='center',
                fontsize=font_annot * APPEARANCE['annotation_font_scale'],
                color='gray'
            )


        # Contour lines

        if show_contours_w.value and cs is not None:

            # Use the same visible interpolated data to draw contour lines
            ax.contour(

                Xi,
                Yi,
                Zi,

                levels=cs.levels[::5],

                colors='k',

                linewidths=lw_contour
            )


        # Bu = 0

        if show_zero_line_w.value:

            ax.axhline(

                native_field(profile_bu_w),

                color=APPEARANCE['zero_line_color'],

                linestyle='--',

                linewidth=APPEARANCE['zero_line_width'],

                alpha=APPEARANCE['zero_line_alpha']
            )


        # Individual colour bars

        if cbar_style_w.value == 'indiv' and cs is not None:

            fmt_str = (
                '%.2f'
                if norm_mode_w.value == 'max_norm'
                else '%.1e'
            )


            cbar = fig.colorbar(

                cs,

                ax=ax,
                use_gridspec=False,

                fraction=0.046,

                pad=0.04,

                format=fmt_str
            )


            cbar.ax.tick_params(

                labelsize=font_tick * 0.85,

                length=2,

                pad=1
            )


        # Panel titles and labels

        if (
            show_sub_labels_w.value
            and idx < len(alphabet)
        ):

            sub_letter = (
                f"({alphabet[idx]}) "
            )

        else:

            sub_letter = ""


        ax.set_title(

            f"{sub_letter}{sample['specimen'] or sample['name']}",

            fontsize=font_title,

            fontweight='bold',

            pad=3,

            loc='left'
        )


        # Axis labels and ticks
        from matplotlib.ticker import MaxNLocator, FuncFormatter
        # Data stay in input units; controls are converted back before plotting.
        tick_factor = 1000.0 if unit_w.value == 'SI' and axis_unit_w.value == 'mT' else 1.0
        def field_tick(value, position, factor=tick_factor):
            scaled = value * factor
            return '0' if abs(scaled) < 1e-12 else f'{scaled:.6g}'
        for axis in (ax.xaxis, ax.yaxis):
            axis.set_major_locator(MaxNLocator(nbins=axis_ticks_w.value,
                                               steps=[1, 2, 2.5, 5, 10]))
            axis.set_major_formatter(FuncFormatter(field_tick))


        # Y axis

        if not any(r * ncols + col in occupied_positions for col in range(c)):

            ax.set_ylabel(

                ylabel,

                fontsize=font_label,

                labelpad=2
            )

        else:

            ax.set_ylabel('')

            if clean_ticks_w.value:

                ax.tick_params(
                    labelleft=False
                )


        # X axis

        if (
            not any(slot in occupied_positions
                    for slot in range(position + ncols, last_position, ncols))
        ):

            ax.set_xlabel(

                xlabel,

                fontsize=font_label,

                labelpad=2
            )

        else:

            ax.set_xlabel('')

            if clean_ticks_w.value:

                ax.tick_params(
                    labelbottom=False
                )


        ax.tick_params(

            axis='both',

            which='major',

            labelsize=font_tick,

            length=3,

            pad=2
        )


        # GLOBAL AXIS LIMITS

        ax.set_xlim([
            native_field(xlim_min_w),
            native_field(xlim_max_w)
        ])


        ax.set_ylim([
            native_field(ylim_min_w),
            native_field(ylim_max_w)
        ])
        ax.set_aspect('equal' if equal_aspect_w.value else 'auto',
                      adjustable='box', anchor='C')


        # ρmax annotation

        if show_rho_max_w.value:

            rho_val = sample['raw_max']

            if np.isfinite(rho_val):
                rho_str = (
                    f"$\\rho_{{\\mathrm{{max}}}}$ = "
                    f"{rho_val:.2e} "
                    f"{raw_unit_str}"
                )
            else:
                rho_str = (
                    f"$\\rho_{{\\mathrm{{max}}}}$ = N/A "
                    f"{raw_unit_str}"
                )


            ax.text(

                *APPEARANCE['annotation_position'],

                rho_str,

                transform=ax.transAxes,

                fontsize=font_annot * APPEARANCE['annotation_font_scale'],

                ha='right',
                va='top',

                bbox=dict(

                    boxstyle='round,pad=0.2',

                    facecolor=APPEARANCE['annotation_background'],

                    alpha=APPEARANCE['annotation_background_alpha'],

                    edgecolor='none'
                )
            )


        # VARIFORC annotations

        if (
            show_variforc_w.value
            and sample.get('variforc')
        ):

            vp = sample['variforc']


            sc0 = vp.get(
                'Sc0',
                '?'
            )

            sc1 = vp.get(
                'Sc1',
                '?'
            )

            su0 = vp.get(
                'Su0',
                '?'
            )

            su1 = vp.get(
                'Su1',
                '?'
            )

            lam = vp.get(
                'lambda',
                '?'
            )


            variforc_str = (

                                f"Sc0={sc0}, Sc1={sc1}\n"

                f"Su0={su0}, Su1={su1}\n"
                f"λ={lam}"
            )


            ax.annotate(
                variforc_str, xy=APPEARANCE['annotation_position'], xycoords='axes fraction',
                xytext=(0, -APPEARANCE['smoothing_gap'] * font_annot * APPEARANCE['annotation_font_scale'] if show_rho_max_w.value else 0),
                textcoords='offset points', fontsize=font_annot * APPEARANCE['annotation_font_scale'],
                ha='right', va='top', linespacing=APPEARANCE['annotation_linespacing'],
                bbox=dict(boxstyle='round,pad=0.20', facecolor=APPEARANCE['annotation_background'],
                          alpha=APPEARANCE['annotation_background_alpha'], edgecolor='none'))


        draw_optional_overlays(ax, sample, hysteresis_curves.get(sample['name']))

    # Unused panels

    for idx in range(nrows * ncols):
        if idx in occupied_positions:
            continue

        r, c = divmod(
            idx,
            ncols
        )

        fig.add_subplot(
            gs[r, c]
        ).axis('off')


    # Shared colour bar

    if (
        cbar_style_w.value == 'global'
        and last_mappable is not None
    ):

        cbar_ax = fig.add_axes([
            0.90,
            0.12,
            0.02,
            0.76
        ])


        fmt_str = (
            '%.2f'
            if norm_mode_w.value == 'max_norm'
            else '%.1e'
        )


        cbar = fig.colorbar(

            last_mappable,

            cax=cbar_ax,

            format=fmt_str
        )


        cbar.set_label(

            zlabel,

            fontsize=font_label,

            fontweight='bold'
        )


        cbar.ax.tick_params(
            labelsize=font_tick
        )


    return fig


# Redraw

def update_plots(*args):
    """Refresh the figure."""

    if not folder_store['loaded']:
        return


    with plot_out:

        plot_out.clear_output(
            wait=True
        )


        fig = generate_multipanel_figure()


        display(fig)


        plt.close(fig)


# Widget callbacks

_field_unit_state = {'factor': field_display_factor()}

def sync_field_input_units(change=None):
    old_factor = _field_unit_state['factor']
    new_factor = field_display_factor()
    was_loaded = folder_store['loaded']
    folder_store['loaded'] = False
    try:
        for widget in (xlim_min_w, xlim_max_w, ylim_min_w, ylim_max_w, profile_bu_w):
            widget.value *= new_factor / old_factor
        _field_unit_state['factor'] = new_factor
        units = axis_unit_w.value if unit_w.value == 'SI' else 'Oe'
        for widget, label in ((xlim_min_w, 'Bc min'), (xlim_max_w, 'Bc max'),
                              (ylim_min_w, 'Bu min'), (ylim_max_w, 'Bu max'),
                              (profile_bu_w, 'Profile Bu')):
            widget.description = f'{label} ({units}):'
    finally:
        folder_store['loaded'] = was_loaded
    update_plots()

axis_unit_w.observe(sync_field_input_units, names='value')
unit_w.observe(sync_field_input_units, names='value')
sync_field_input_units()

for w in [

    ncols_w,

    axis_ticks_w,

    cmap_w,

    scale_mode_w,

    norm_mode_w,

    show_rho_max_w,

    # VARIFORC
    show_variforc_w,

    show_sub_labels_w,

    show_zero_line_w,

    clean_ticks_w,

    cbar_style_w,

    levels_w,

    show_contours_w,

    xlim_min_w,

    xlim_max_w,

    ylim_min_w,

    ylim_max_w,
    equal_aspect_w, show_profile_w, profile_bu_w, profile_height_w, show_hysteresis_w, hysteresis_size_w,
    show_hysteresis_axes_w, hysteresis_color_w, forc_hysteresis_color_w, hysteresis_dir_w, hysteresis_suffix_w,
    hysteresis_match_prefix_w, hysteresis_source_w, forc_hysteresis_dir_w,
    show_hysteresis_range_w, hysteresis_field_unit_w
]:

    w.observe(
        update_plots,
        names='value'
    )


# Export

def save_multipanel_plot(b):
    """Export the current figure as PDF, PNG or SVG."""

    if not folder_store['loaded']:

        status_msg.value = (

            "<span style='color:red;'>"

            "Load a FORC folder first."

            "</span>"
        )

        return


    # Blank output folder: use the input folder.
    custom_folder = save_dir_w.value.strip()

    if custom_folder:
        out_folder = os.path.abspath(
            os.path.expanduser(custom_folder)
        )
    else:
        out_folder = folder_store['folder_path']

    if not os.path.isdir(out_folder):
        status_msg.value = (
            "<span style='color:red;'>"
            "Output folder not found: "
            f"<b>{out_folder}</b>"
            "</span>"
        )
        return

    # The selected format determines the extension.
    custom_name = save_name_w.value.strip()

    if custom_name:
        custom_name = os.path.basename(custom_name)
        name_root, _ = os.path.splitext(custom_name)
        if not name_root:
            name_root = custom_name
        filename = f"{name_root}.{fmt_w.value}"
    else:
        filename = (
            f"FORC_Multipanel_"
            f"{ncols_w.value}cols."
            f"{fmt_w.value}"
        )

    out_path = os.path.join(
        out_folder,
        filename
    )


    fig = generate_multipanel_figure()


    fig.savefig(

        out_path,

        dpi=dpi_w.value,

        bbox_inches='tight'
    )


    plt.close(fig)


    status_msg.value = (

        "<span style='color:green;'>"

        "Saved:<br>"

        f"<b>{out_path}</b>"

        "</span>"
    )


save_btn.on_click(
    save_multipanel_plot
)


# Dashboard

ui_top = widgets.VBox([

    # Folder selection

    dir_chooser,

    dirpath_w,


    widgets.HBox(

        [
            load_btn,
            unit_w,
            grid_res_w
        ],

        layout=widgets.Layout(
            align_items='center',
            gap='10px'
        )
    ),


    # Grid / normalization

    widgets.HTML(

        "<hr style='margin:5px 0;'>"

        "<b>Layout and intensity</b>"
    ),


    widgets.HBox(

        [
            ncols_w,
            norm_mode_w,
            show_rho_max_w,
            cbar_style_w
        ],

        layout=widgets.Layout(
            align_items='center',
            gap='10px'
        )
    ),


    # Annotations

    widgets.HBox(

        [
            show_sub_labels_w,
            show_zero_line_w,
            clean_ticks_w,
            show_contours_w,

            # VARIFORC
            show_variforc_w
        ],

        layout=widgets.Layout(
            align_items='center',
            gap='10px'
        )
    ),


    specimen_panel,

    # Colour options

    widgets.HBox(

        [
            cmap_w,
            scale_mode_w,
            levels_w
        ],

        layout=widgets.Layout(
            align_items='center',
            gap='10px'
        )
    ),


    # Axis limits and equal physical field scale
    widgets.HBox([axis_unit_w, axis_ticks_w]),
    axis_units_note,
    axes_tabs,
    insets_ui
])


# Export controls

ui_bottom = widgets.VBox([

    widgets.HTML(

        "<hr>"

        "<b>Export</b>"
    ),


    widgets.HBox(

        [
            save_dir_w
        ],

        layout=widgets.Layout(
            align_items='center',
            gap='10px'
        )
    ),

    widgets.HBox(

        [
            save_name_w,
            fmt_w,
            dpi_w,
            save_btn
        ],

        layout=widgets.Layout(
            align_items='center',
            gap='12px'
        )
    ),


    status_msg
])


# Display

dashboard = widgets.VBox([

    widgets.HTML(

        "<h2>FORC multipanel</h2>"
    ),

    ui_top,

    widgets.HTML("<hr>"),

    plot_out,

    ui_bottom
])


display(dashboard)
